<a href="https://colab.research.google.com/github/srivastava071/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip -q install duckdb huggingface_hub scikit-learn

In [ ]:
import os
import getpass

HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

HF_TOKEN = HF_TOKEN or getpass.getpass(
    "Paste your Hugging Face READ token (hf_...): "
)

print("Hugging Face token found:", bool(HF_TOKEN))

Paste your Hugging Face READ token (hf_...): ··········
Hugging Face token found: True


In [ ]:
import duckdb

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

print("DuckDB connected.")

DuckDB connected.


In [ ]:
REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_query_90d": f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

## Step 1. Build the feature vector

I use five historical numerical features from the content-refresh lane:

- previous_30d_impressions
- previous_30d_clicks
- previous_30d_avg_position
- visible_query_count
- top_query_share

The client and content hash IDs are kept only as identifiers and are not used as predictive features. Therefore, no categorical encoding is required.

Missing numerical values are filled using the median of the corresponding feature. The median is calculated from the feature data available in the training frame.

All five features are intended to be available before the decision moment.

Step 2 — Build the actual feature vector

In [ ]:
# Build historical features for March 2026 decisions

features = con.sql(f"""
WITH daily AS (
    SELECT
        client_hash_id,
        content_hash_id,
        report_date,
        gsc_impressions,
        gsc_clicks,
        gsc_avg_position
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '2026-02-01'
      AND report_date < DATE '2026-04-01'
),

march_pages AS (
    SELECT DISTINCT
        client_hash_id,
        content_hash_id
    FROM daily
    WHERE report_date >= DATE '2026-03-01'
      AND report_date < DATE '2026-04-01'
),

historical_features AS (
    SELECT
        m.client_hash_id,
        m.content_hash_id,

        SUM(
            CASE
                WHEN d.report_date >= DATE '2026-02-01'
                 AND d.report_date < DATE '2026-03-01'
                THEN COALESCE(d.gsc_impressions, 0)
                ELSE 0
            END
        ) AS previous_30d_impressions,

        SUM(
            CASE
                WHEN d.report_date >= DATE '2026-02-01'
                 AND d.report_date < DATE '2026-03-01'
                THEN COALESCE(d.gsc_clicks, 0)
                ELSE 0
            END
        ) AS previous_30d_clicks,

        AVG(
            CASE
                WHEN d.report_date >= DATE '2026-02-01'
                 AND d.report_date < DATE '2026-03-01'
                THEN d.gsc_avg_position
            END
        ) AS previous_30d_avg_position

    FROM march_pages m
    LEFT JOIN daily d
      ON m.client_hash_id = d.client_hash_id
     AND m.content_hash_id = d.content_hash_id

    GROUP BY
        m.client_hash_id,
        m.content_hash_id
),

query_features AS (
    SELECT
        content_hash_id,
        ANY_VALUE(content_visible_query_count) AS visible_query_count,
        MAX(impressions_90d) AS top_query_impressions,
        SUM(impressions_90d) AS total_query_impressions
    FROM {TABLES['fact_query_90d']}
    GROUP BY content_hash_id
)

SELECT
    h.client_hash_id,
    h.content_hash_id,
    h.previous_30d_impressions,
    h.previous_30d_clicks,
    h.previous_30d_avg_position,
    q.visible_query_count,
    q.top_query_impressions / NULLIF(q.total_query_impressions, 0)
        AS top_query_share

FROM historical_features h
LEFT JOIN query_features q
    ON h.content_hash_id = q.content_hash_id
""").df()

print("Raw feature frame shape:", features.shape)
features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Raw feature frame shape: (331437, 7)


,client_hash_id,content_hash_id,previous_30d_impressions,previous_30d_clicks,previous_30d_avg_position,visible_query_count,top_query_share
0,client_08a6a72ff48e62c0,content_5f6fae04728d32ab,324.0,0.0,5.337775,5,0.377622
1,client_08a6a72ff48e62c0,content_5f71205e0b46f70a,1574.0,3.0,2.909252,12,0.306488
2,client_08a6a72ff48e62c0,content_5f716493f7989b45,510.0,2.0,7.437261,17,0.213628
3,client_08a6a72ff48e62c0,content_5f8b67a6b0494e15,26.0,0.0,10.275510,1,1.000000
4,client_08a6a72ff48e62c0,content_5f93f846d9834caf,12.0,0.0,8.857143,2,0.560000


Step 3 — Fill missing values

In [ ]:
feature_cols = [
    "previous_30d_impressions",
    "previous_30d_clicks",
    "previous_30d_avg_position",
    "visible_query_count",
    "top_query_share"
]

# Show missing values before filling
print("Missing values before filling:")
print(features[feature_cols].isna().sum())

# Fill numerical missing values with the median
for col in feature_cols:
    features[col] = features[col].fillna(features[col].median())

print("\nMissing values after filling:")
print(features[feature_cols].isna().sum())

Missing values before filling:
previous_30d_impressions     0
previous_30d_clicks          0
previous_30d_avg_position    0
visible_query_count          0
top_query_share              0
dtype: int64

Missing values after filling:
previous_30d_impressions     0
previous_30d_clicks          0
previous_30d_avg_position    0
visible_query_count          0
top_query_share              0
dtype: int64


Step 4 — Create the actual ML vector

In [ ]:
X = features[feature_cols].copy()

print("Feature vector shape:", X.shape)
print("Features:")
print(X.columns.tolist())

X.head()

Feature vector shape: (331437, 5)
Features:
['previous_30d_impressions', 'previous_30d_clicks', 'previous_30d_avg_position', 'visible_query_count', 'top_query_share']


,previous_30d_impressions,previous_30d_clicks,previous_30d_avg_position,visible_query_count,top_query_share
0,324.0,0.0,5.337775,5,0.377622
1,1574.0,3.0,2.909252,12,0.306488
2,510.0,2.0,7.437261,17,0.213628
3,26.0,0.0,10.275510,1,1.000000
4,12.0,0.0,8.857143,2,0.560000


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

Step 2.1
### 1. Previous 30-day impressions
This represents the total search impressions received by the content item during the previous 30 days. Missing values are filled with the median value of the feature. It is available before the decision moment because it comes from historical data.

### 2. Previous 30-day clicks
This represents the total search clicks received by the content item during the previous 30 days. Missing values are filled with the median value of the feature. It is available before the decision moment because the clicks have already been recorded.

### 3. Previous 30-day average position
This represents the average search position of the content item during the previous 30 days. Missing values are filled with the median value of the feature. It is available before the decision moment because it is calculated from historical ranking observations.

### 4. Visible query count
This represents the number of visible queries associated with the content item in the query-level data. Missing values are filled with the median value of the feature. It is used as historical query information available before the decision moment.

### 5. Top query share
This represents the share of retained query impressions coming from the highest-impression query for the content item. Missing values are filled with the median value of the feature. It is based on historical query information and is intended to be available before the decision moment.

All five features are numerical, so categorical encoding is not required. No client or content hash is used as a predictive feature.

Step 2.2 — Added a small verification code cell

In [ ]:
print("Feature notes check")
print("=" * 40)

for col in feature_cols:
    print(f"{col}:")
    print(f"  Missing values: {features[col].isna().sum()}")
    print(f"  Data type: {features[col].dtype}")
    print()

Feature notes check
previous_30d_impressions:
  Missing values: 0
  Data type: float64

previous_30d_clicks:
  Missing values: 0
  Data type: float64

previous_30d_avg_position:
  Missing values: 0
  Data type: float64

visible_query_count:
  Missing values: 0
  Data type: Int64

top_query_share:
  Missing values: 0
  Data type: float64



Step 2.3 — Added an availability check

In [ ]:
availability_check = {
    "previous_30d_impressions": "Historical data before March 2026",
    "previous_30d_clicks": "Historical data before March 2026",
    "previous_30d_avg_position": "Historical data before March 2026",
    "visible_query_count": "Historical query data",
    "top_query_share": "Historical query data"
}

for feature, timing in availability_check.items():
    print(f"{feature}: {timing}")

previous_30d_impressions: Historical data before March 2026
previous_30d_clicks: Historical data before March 2026
previous_30d_avg_position: Historical data before March 2026
visible_query_count: Historical query data
top_query_share: Historical query data


## 3.1

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*


## 3. The leakage hunt

I checked my feature vector for information that would not be available at the decision moment.

The main leakage risk is `is_declining_label`. This field represents the outcome I am trying to predict, so using it as a feature would give the model the answer directly.

I also checked for future-window information. My five features are calculated from historical data before the March 2026 decision window, so the March outcome is not used to create the five-feature vector.

I did not use FlyRank product decision fields such as priority scores or action flags. The internship data is intended to contain observable signals rather than the product's own decisions. Using a product decision as a feature would create a circular result because the model would simply learn an existing decision.


3.2 First, honest feature set

In [ ]:
print("Honest feature set:")
for feature in feature_cols:
    print("-", feature)

Honest feature set:
- previous_30d_impressions
- previous_30d_clicks
- previous_30d_avg_position
- visible_query_count
- top_query_share


3.3 leaking label

In [ ]:
leaky_X = X.copy()

# Deliberate leakage:
# the target itself is being added as an input feature
leaky_X["is_declining_label"] = 0

print("Honest feature count:", len(feature_cols))
print("Leaky feature count:", leaky_X.shape[1])
print("Added leakage feature: is_declining_label")

Honest feature count: 5
Leaky feature count: 6
Added leakage feature: is_declining_label


3.4 Create the label

In [ ]:
outcomes = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_impressions) AS march_impressions
FROM {TABLES['fact_daily']}
WHERE report_date >= DATE '2026-03-01'
  AND report_date < DATE '2026-04-01'
GROUP BY
    client_hash_id,
    content_hash_id
""").df()

features_with_label = features.merge(
    outcomes,
    on=["client_hash_id", "content_hash_id"],
    how="left"
)

features_with_label["is_declining_label"] = (
    features_with_label["march_impressions"]
    < 0.8 * features_with_label["previous_30d_impressions"]
).astype(int)

print(features_with_label["is_declining_label"].value_counts())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

is_declining_label
0    293571
1     37866
Name: count, dtype: int64


3.5 perform the leakage test

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

In [ ]:
leak_data = features_with_label.dropna(
    subset=feature_cols + ["is_declining_label"]
).copy()

leak_features = feature_cols + ["is_declining_label"]

X_leak = leak_data[leak_features]
y_leak = leak_data["is_declining_label"]

X_train, X_test, y_train, y_test = train_test_split(
    X_leak,
    y_leak,
    test_size=0.25,
    random_state=42,
    stratify=y_leak
)

leak_model = DecisionTreeClassifier(
    max_depth=3,
    random_state=42
)

leak_model.fit(X_train, y_train)

leak_predictions = leak_model.predict(X_test)

leak_accuracy = accuracy_score(
    y_test,
    leak_predictions
)

print("Leakage-test accuracy:", round(leak_accuracy, 3))

Leakage-test accuracy: 1.0


3.6

### Leakage result

The leakage test produced an unrealistically strong result because `is_declining_label` was included as an input feature.

This feature contains the answer that the model is supposed to predict. Therefore, the high score does not represent genuine predictive ability.

This confirms that label-derived information must not be included in the honest feature vector.

I keep only the five historical features in the final feature vector.

3.7 Finally, prove the leaked feature is removed


In [ ]:
honest_X = features[feature_cols].copy()

print("Final honest feature vector:")
print(honest_X.columns.tolist())

print("\nNumber of final features:", honest_X.shape[1])

Final honest feature vector:
['previous_30d_impressions', 'previous_30d_clicks', 'previous_30d_avg_position', 'visible_query_count', 'top_query_share']

Number of final features: 5


## 4. What I excluded and why

I deliberately excluded the following fields from my final feature vector:

- `is_declining_label` — excluded because it is the target I am trying to predict. Using it as a feature would directly leak the answer into the model.

- March 2026 outcome information — excluded from the historical feature vector because it represents the period being evaluated and would not be available at the earlier decision moment.

- `client_hash_id` — kept only as an identifier for grouping and analysis, not as a predictive feature. Using the ID as a feature could allow the model to memorise client-specific patterns rather than learn useful search signals.

- `content_hash_id` — kept only as an identifier for the content item and not used as a predictive feature for the same reason.

- Product decision fields such as priority or action flags — excluded because they represent decisions that could already contain the answer we are trying to learn. Including them could make the model circular.

- Any future or label-derived information — excluded because it would not be available at the decision moment and could create misleadingly strong results.

My final feature vector therefore contains only five historical search-intelligence features that are intended to be available before the decision moment.

In [ ]:
excluded_features = {
    "is_declining_label": "Target leakage",
    "March 2026 outcome data": "Future/outcome information",
    "client_hash_id": "Identifier, not a predictive signal",
    "content_hash_id": "Identifier, not a predictive signal",
    "product decision fields": "Could create circular/copycat predictions",
    "future or label-derived fields": "Not available at decision time"
}

for field, reason in excluded_features.items():
    print(f"{field} -> {reason}")

is_declining_label -> Target leakage
March 2026 outcome data -> Future/outcome information
client_hash_id -> Identifier, not a predictive signal
content_hash_id -> Identifier, not a predictive signal
product decision fields -> Could create circular/copycat predictions
future or label-derived fields -> Not available at decision time


In [ ]:
print("Final feature vector:")
print(honest_X.columns.tolist())

print("\nNumber of features:", honest_X.shape[1])

Final feature vector:
['previous_30d_impressions', 'previous_30d_clicks', 'previous_30d_avg_position', 'visible_query_count', 'top_query_share']

Number of features: 5


## Self-check

Before you submit, confirm each line honestly:

☑ Every section above is filled — markdown thinking AND the code that backs it

☑ The notebook runs top to bottom with no errors

☑ No client names, URLs, or private queries anywhere

☑ My claims use careful words: observed, measured, directional, decision-support

☑ Committed to my repo under work/notebooks/